In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
from jax import tree_util as jtu
from jax import numpy as jnp, lax
import numpy as np

In [ ]:
params = {
    "contact_rate": 1.0,
    "recovery_rate": 2.0,
    "seed_start": 20.0,
    "seed_rate": 2.0
}

In [ ]:
treedef = jtu.tree_structure(params)
treedef

In [ ]:
flat_params, treedef = jtu.tree_flatten(params)

In [ ]:
jtu.tree_unflatten(treedef, flat_params)

In [ ]:
def seed_strain(t,seed_start):
    #rate = t-seed_start
    return jnp.where((t >= seed_start) & (t < (seed_start + 7.0)), 1.0, 0.0)

In [ ]:
mm = np.exp(np.random.normal(0.0,0.2,size=(128,128))) / 128

In [ ]:
def vector_field(t, y, args):
    params: dict = jtu.tree_unflatten(treedef, args)
    s = y[0:128]
    i = y[128:256]
    r = y[256:(256+128)]
    imports = seed_strain(t, params["seed_start"])
    i_props = i / y.reshape(3,128).sum(axis=0)
    infections = s * (mm @ i_props) * params["contact_rate"]
    #infections = s * (jnp.sum(i)/(jnp.sum(y))) * params["contact_rate"]
    recoveries = i * params["recovery_rate"]
    ds = -infections
    di = infections - recoveries + imports * params["seed_rate"]
    dr = recoveries
    return jnp.concat([ds,di,dr])

In [ ]:
params = {
    "contact_rate": 1.0,
    "recovery_rate": 2.0,
    "seed_start": 20.0,
    "seed_rate": 0.01,
}

iy = jnp.concat([
    jnp.ones(128)* 100.0,
    jnp.zeros(128),
    jnp.zeros(128)
]
)

flat_params, treedef = jtu.tree_flatten(params)

vector_field(30.2, iy, flat_params)[128]


In [ ]:
from diffrax import diffeqsolve, Dopri5, ODETerm, SaveAt, PIDController
import diffrax



#print(sol.ts)  # DeviceArray([0.   , 1.   , 2.   , 3.    ])
#print(sol.ys)  # DeviceArray([1.   , 0.368, 0.135, 0.0498])

In [ ]:
import pandas as pd
from equinox import filter_jit

In [ ]:

@filter_jit
def solve_split(params):

    iy = jnp.concat([
        jnp.ones(128) * 100.0,
        jnp.zeros(128),
        jnp.zeros(128)
    ]
    )

    flat_params, treedef = jtu.tree_flatten(params)

    term = ODETerm(vector_field)
    solver = Dopri5()
    saveat = SaveAt(ts=jnp.arange(201.0), fn = lambda t, y, args: y[jnp.array([0,128,256])])
    stepsize_controller = PIDController(rtol=1e-5, atol=1e-5, step_ts=[params["seed_start"], params["seed_start"] + 7.0])

    #stepsize_controller = diffrax.ConstantStepSize()
    adjoint=diffrax.RecursiveCheckpointAdjoint(checkpoints=400)
    #adjoint = diffrax.ForwardMode()
    #adjoint = diffrax.DirectAdjoint()
    sol = diffeqsolve(term, solver, t0=0, t1=201.0, throw=False, max_steps=400, dt0=0.1, y0=iy, args = flat_params, saveat=saveat,
                    stepsize_controller=stepsize_controller, adjoint=adjoint)

    #solb = diffeqsolve(term, solver, t0=inflection, t1=100.0, dt0=0.1, y0=sola.ys[-1], args = flat_params, saveat=saveat,
    #                stepsize_controller=stepsize_controller)
    
    return sol#a, solb

#ydf = pd.DataFrame(sol.ys)

In [ ]:
params = {
    "contact_rate": jnp.linspace(0.01,0.2,128),
    "recovery_rate": jnp.linspace(0.01,0.02,128),
    "seed_start": 12.3,
    "seed_rate": 1.0,
}

sol = solve_split(params)
sol.stats

target = sol.ys[::16]
tsol = sol.ys

In [ ]:
sol.stats

In [ ]:
pd.DataFrame(sol.ys).plot()

In [ ]:
pd.DataFrame(target).plot(legend=False)

In [ ]:
def ll(params):
    sol = solve_split(params)
    ys = jnp.where(jnp.isfinite(sol.ts), sol.ys.T, 0.0).T[::16]
    diff = ys-target
    return jnp.sqrt((diff*diff).sum())
    #successful = (sol.result == diffrax.RESULTS.successful)
    #jnp.where(~jnp.isfinite(sol.ts), 1.0, 0.0)
    #return jnp.where(successful, jnp.sqrt(((sol.ys - target)**2).sum()), 1e100)
    

In [ ]:
gll = jax.jit(jax.grad(ll))

In [ ]:
import jax.flatten_util
from jax.flatten_util import ravel_pytree

opt_params = {
    "contact_rate": jnp.ones(128) * 0.3,
    "recovery_rate": jnp.ones(128) * 0.1,
    "seed_start": 30.0,
    "seed_rate": 0.1,
}

opt_params_flat, unravel = jax.flatten_util.ravel_pytree(opt_params)

@jax.jit
def ll_flat(flat_params):
    params = unravel(flat_params)
    return ll(params)

In [ ]:
gll_flat = jax.jit(jax.grad(ll_flat))

In [ ]:
def ll_args(params, args):
    return ll(params)

In [ ]:
import optimistix as optx

In [ ]:
def residuals(params, values):
    sol = solve_split(params)
    ys = jnp.where(jnp.isfinite(sol.ts), sol.ys.T, 1e10).T[::16]
    return values - ys

In [ ]:
values = target
solver = optx.LevenbergMarquardt(
    rtol=1e-8, atol=1e-8, verbose=frozenset({"step", "accepted", "loss", "step_size"})
)

init_parameters = opt_params
sol = optx.least_squares(residuals, solver, init_parameters, args=values)

In [ ]:
import scipy
import scipy.optimize

In [ ]:
res = scipy.optimize.minimize(ll_flat, opt_params_flat, jac=gll_flat,method="L-BFGS-B")

In [ ]:
res

In [ ]:
unravel(res.x)

In [ ]:
ll_flat(res.x)

In [ ]:
pd.DataFrame(target).plot(legend=False)

In [ ]:
pd.DataFrame(solve_split(unravel(res.x)).ys[::16]).plot(legend=False)

In [ ]:
sol = solve_split(opt_params)

In [ ]:
ydf = pd.DataFrame(sol.ys-target)